In [1]:
# 05B-2. 환경 설정
# ECOS Macro Pipeline 기본 라이브러리 및 경로 설정

from pathlib import Path
import os

import pandas as pd
import requests

from dotenv import load_dotenv


PROJECT_ROOT = Path(
    r"C:\code\portfolio_optimization"
)

ECOS_RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ecos"
)

ECOS_RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

load_dotenv(
    PROJECT_ROOT / ".env"
)

ECOS_API_KEY = os.getenv(
    "ECOS_API_KEY"
)

print(
    "Project root:",
    PROJECT_ROOT
)

print(
    "ECOS API key loaded:",
    bool(ECOS_API_KEY)
)

Project root: C:\code\portfolio_optimization
ECOS API key loaded: True


In [2]:
#05b-3. ecos api 기본 조회 함수
# 원천 데이터 수집 준비
# ECOS StatisticSearch 공통 조회 함수 생성

# 목적:
# - 환율 / 금리 등 여러 ECOS 통계를 동일한 방식으로 조회
# - API 오류를 명확하게 확인 

ECOS_BASE_URL = (
    "https://ecos.bok.or.kr"
    "/api/StatisticSearch"
)


def fetch_ecos_series(
    stat_code,
    frequency,
    start_date,
    end_date,
    item_code,
    start_index=1,
    end_index=10000
):

    url = (
        f"{ECOS_BASE_URL}"
        f"/{ECOS_API_KEY}"
        f"/json"
        f"/kr"
        f"/{start_index}"
        f"/{end_index}"
        f"/{stat_code}"
        f"/{frequency}"
        f"/{start_date}"
        f"/{end_date}"
        f"/{item_code}"
    )

    response = requests.get(
        url,
        timeout=30
    )

    print(
        "HTTP status:",
        response.status_code
    )

    response.raise_for_status()

    data = response.json()

    # ECOS 자체 오류 응답 처리
    if "RESULT" in data:

        raise RuntimeError(
            data["RESULT"]
        )

    rows = (
        data
        .get("StatisticSearch", {})
        .get("row", [])
    )

    return pd.DataFrame(
        rows
    )

In [3]:
#05b-4. 기준금리 test

# STAT_CODE  = 통계표 코드
# ITEM_CODE  = 통계 항목 코드
# D          = Daily, 일별

base_rate_test = fetch_ecos_series(
    stat_code="722Y001",
    frequency="D",
    start_date="20260101",
    end_date="20260915",
    item_code="0101000"
)

print(
    "Shape:",
    base_rate_test.shape
)

base_rate_test.head()

HTTP status: 200
Shape: (258, 14)


,STAT_CODE,STAT_NAME,ITEM_CODE1,ITEM_NAME1,ITEM_CODE2,ITEM_NAME2,ITEM_CODE3,ITEM_NAME3,ITEM_CODE4,ITEM_NAME4,UNIT_NAME,WGT,TIME,DATA_VALUE
0,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260101,2.5
1,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260102,2.5
2,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260103,2.5
3,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260104,2.5
4,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260105,2.5


In [4]:
#05b-5.기준금리 API 응답 필드 확인

print(
    base_rate_test.columns.tolist()
)

base_rate_test[
    [
        "STAT_CODE",
        "STAT_NAME",
        "ITEM_CODE1",
        "ITEM_NAME1",
        "UNIT_NAME",
        "TIME",
        "DATA_VALUE"
    ]
].head(10)

['STAT_CODE', 'STAT_NAME', 'ITEM_CODE1', 'ITEM_NAME1', 'ITEM_CODE2', 'ITEM_NAME2', 'ITEM_CODE3', 'ITEM_NAME3', 'ITEM_CODE4', 'ITEM_NAME4', 'UNIT_NAME', 'WGT', 'TIME', 'DATA_VALUE']


,STAT_CODE,STAT_NAME,ITEM_CODE1,ITEM_NAME1,UNIT_NAME,TIME,DATA_VALUE
0,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260101,2.5
1,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260102,2.5
2,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260103,2.5
3,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260104,2.5
4,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260105,2.5
5,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260106,2.5
6,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260107,2.5
7,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260108,2.5
8,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260109,2.5
9,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,연%,20260110,2.5


In [5]:
#05b-6. 전체 연구기간 기준금리 수집
#연구기간: 2018-02-05~2026-09-15

base_rate_raw = fetch_ecos_series(
    stat_code="722Y001",
    frequency="D",
    start_date="20180205",
    end_date="20260915",
    item_code="0101000"
)

print(
    "Shape:",
    base_rate_raw.shape
)

base_rate_raw.tail()

HTTP status: 200
Shape: (3145, 14)


,STAT_CODE,STAT_NAME,ITEM_CODE1,ITEM_NAME1,ITEM_CODE2,ITEM_NAME2,ITEM_CODE3,ITEM_NAME3,ITEM_CODE4,ITEM_NAME4,UNIT_NAME,WGT,TIME,DATA_VALUE
3140,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260911,3
3141,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260912,3
3142,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260913,3
3143,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260914,3
3144,722Y001,1.3.1. 한국은행 기준금리 및 여수신금리,0101000,한국은행 기준금리,None,None,None,None,None,None,연%,None,20260915,3


In [6]:
#05b-7. raw 기준금리 raw data 저장
#raw layer:
# - 컬럼명 변경 전
# - 값 변환 전
# - API 원본 형태 보존

BASE_RATE_RAW_PATH = (
    ECOS_RAW_DIR
    / "bok_base_rate_raw.csv"
)

base_rate_raw.to_csv(
    BASE_RATE_RAW_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    BASE_RATE_RAW_PATH
)

Saved: C:\code\portfolio_optimization\data\raw\ecos\bok_base_rate_raw.csv


In [7]:
#05b-8. 기준금리 데이터 정제
#필요한 컬럼만 선택, 타입 변환

#time -> date
#data_value -> base_rate

base_rate_clean = (
    base_rate_raw[
        [
            "TIME",
            "DATA_VALUE"
        ]
    ]
    .copy()
)

base_rate_clean = (
    base_rate_clean
    .rename(
        columns={
            "TIME": "date",
            "DATA_VALUE": "base_rate"
        }
    )
)

base_rate_clean["date"] = pd.to_datetime(
    base_rate_clean["date"],
    format="%Y%m%d"
)

base_rate_clean["base_rate"] = pd.to_numeric(
    base_rate_clean["base_rate"],
    errors="coerce" #숫자로 못 바꾸는 값 -> 결측값으로 
)

base_rate_clean = (
    base_rate_clean
    .sort_values("date")
    .reset_index(drop=True)
)

base_rate_clean.head()

,date,base_rate
0,2018-02-05,1.5
1,2018-02-06,1.5
2,2018-02-07,1.5
3,2018-02-08,1.5
4,2018-02-09,1.5


In [8]:
#05b-9. 기준금리 정제 결과 확인
#목적: 기간, 결측치, 중복날짜 확인

print(
    "Start:",
    base_rate_clean["date"].min()
)

print(
    "End:",
    base_rate_clean["date"].max()
)

print(
    "Rows:",
    len(base_rate_clean)
)

print(
    "Missing values:",
    base_rate_clean["base_rate"].isna().sum()
)

print(
    "Duplicate dates:",
    base_rate_clean["date"].duplicated().sum()
)

base_rate_clean.tail()

Start: 2018-02-05 00:00:00
End: 2026-09-15 00:00:00
Rows: 3145
Missing values: 0
Duplicate dates: 0


,date,base_rate
3140,2026-09-11,3.0
3141,2026-09-12,3.0
3142,2026-09-13,3.0
3143,2026-09-14,3.0
3144,2026-09-15,3.0


In [9]:
#05b-10. 기준금리 clean data 저장
#목적: raw data와 정제 데이터를 분리해서 관리, 이후 05d common dataset에서 사용

ECOS_CLEAN_DIR = (
    PROJECT_ROOT
    / "data"
    / "clean"
    / "ecos"
)

ECOS_CLEAN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

BASE_RATE_CLEAN_PATH = (
    ECOS_CLEAN_DIR
    / "bok_base_rate_clean.csv"
)

base_rate_clean.to_csv(
    BASE_RATE_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    BASE_RATE_CLEAN_PATH
)

Saved: C:\code\portfolio_optimization\data\clean\ecos\bok_base_rate_clean.csv


In [10]:
#05c-11. ecos statistic item list 조회 함수
#목적: 통계표 내부 실제  item_code 확인, 항목코드 추측x 

def fetch_ecos_item_list(
    stat_code,
    end_index=1000
):

    url = (
        "https://ecos.bok.or.kr"
        f"/api/StatisticItemList"
        f"/{ECOS_API_KEY}"
        f"/json"
        f"/kr"
        f"/1"
        f"/{end_index}"
        f"/{stat_code}"
    )

    response = requests.get(
        url,
        timeout=30
    )

    print(
        "HTTP status:",
        response.status_code
    )

    response.raise_for_status()

    data = response.json()

    if "RESULT" in data:
        raise RuntimeError(
            data["RESULT"]
        )

    rows = (
        data
        .get("StatisticItemList", {})
        .get("row", [])
    )

    return pd.DataFrame(
        rows
    )

In [12]:
#05b-12. usd/krw 항목 확인
#예상: stat_code=731Y003, item_name = 원/달러 관련 항목

fx_items = fetch_ecos_item_list(
    stat_code="731Y003"
)

print(
    "Shape:",
    fx_items.shape
)

fx_items[
    [
        "ITEM_CODE",
        "ITEM_NAME",
        "CYCLE",
        "START_TIME",
        "END_TIME",
        "UNIT_NAME"
    ]
    ]

HTTP status: 200
Shape: (10, 14)


,ITEM_CODE,ITEM_NAME,CYCLE,START_TIME,END_TIME,UNIT_NAME
0,0000002,원/달러(시가),D,19900302,20260917,원
1,0000005,원/달러(고가),D,19900302,20260916,원
2,0000004,원/달러(저가),D,19900302,20260916,원
3,0000003,원/달러(종가 15:30),D,19900302,20260916,원
4,0000013,원/달러(종가),D,20240701,20260916,원
5,0000007,원/위안(시가),D,20141201,20260916,원
6,0000008,원/위안(고가),D,20141201,20260916,원
7,0000009,원/위안(저가),D,20141201,20260916,원
8,0000010,원/위안(종가),D,20141201,20260916,원
9,0000006,원/100엔(하나은행고시),D,20050302,20260916,원


In [13]:
#05b-13. 국고채 3년물 통계 확인
#예상: stat_code = 817Y002, item_name = 국고채(3년)

bond_items = fetch_ecos_item_list(
    stat_code="817Y002"
)

bond_items[
    bond_items["ITEM_NAME"].str.contains(
        "국고채",
        na=False
    )
][
    [
        "ITEM_CODE",
        "ITEM_NAME",
        "CYCLE",
        "START_TIME",
        "END_TIME",
        "UNIT_NAME"
    ]
]

HTTP status: 200


,ITEM_CODE,ITEM_NAME,CYCLE,START_TIME,END_TIME,UNIT_NAME
2,010200001,국고채(5년),D,20000104,20260916,연%
7,010190000,국고채(1년),D,20000201,20260916,연%
8,010210000,국고채(10년),D,20001218,20260916,연%
9,010220000,국고채(20년),D,20060125,20260916,연%
10,010230000,국고채(30년),D,20120911,20260916,연%
11,010195000,국고채(2년),D,20210310,20260916,연%
12,010200000,국고채(3년),D,19981113,20260916,연%
25,010240000,국고채(50년),D,20161011,20260916,연%


In [14]:
#05b-14. usd/krw 전체 기간 수집
#기간: 2018-02-05~2026-09-15

usdkrw_raw = fetch_ecos_series(
    stat_code="731Y003",
    frequency="D",
    start_date="20180205",
    end_date="20260915",
    item_code="0000003"
)

print(
    "Shape:",
    usdkrw_raw.shape
)

usdkrw_raw.head()

HTTP status: 200
Shape: (2114, 14)


,STAT_CODE,STAT_NAME,ITEM_CODE1,ITEM_NAME1,ITEM_CODE2,ITEM_NAME2,ITEM_CODE3,ITEM_NAME3,ITEM_CODE4,ITEM_NAME4,UNIT_NAME,WGT,TIME,DATA_VALUE
0,731Y003,"3.1.1.3. 원화의 대미달러, 원화의 대위안/대엔 환율",0000003,원/달러(종가 15:30),None,None,None,None,None,None,원,None,20180205,1088.5
1,731Y003,"3.1.1.3. 원화의 대미달러, 원화의 대위안/대엔 환율",0000003,원/달러(종가 15:30),None,None,None,None,None,None,원,None,20180206,1091.5
2,731Y003,"3.1.1.3. 원화의 대미달러, 원화의 대위안/대엔 환율",0000003,원/달러(종가 15:30),None,None,None,None,None,None,원,None,20180207,1086.6
3,731Y003,"3.1.1.3. 원화의 대미달러, 원화의 대위안/대엔 환율",0000003,원/달러(종가 15:30),None,None,None,None,None,None,원,None,20180208,1087.9
4,731Y003,"3.1.1.3. 원화의 대미달러, 원화의 대위안/대엔 환율",0000003,원/달러(종가 15:30),None,None,None,None,None,None,원,None,20180209,1092.1


In [15]:
#05b-15. 원천 데이터 수집4 / 국고채 3년물 금리 전체 연구기간 수집
#단위: 연 %

bond3y_raw = fetch_ecos_series(
    stat_code="817Y002",
    frequency="D",
    start_date="20180205",
    end_date="20260915",
    item_code="010200000"
)

print(
    "Shape:",
    bond3y_raw.shape
)

bond3y_raw.head()

HTTP status: 200
Shape: (2120, 14)


,STAT_CODE,STAT_NAME,ITEM_CODE1,ITEM_NAME1,ITEM_CODE2,ITEM_NAME2,ITEM_CODE3,ITEM_NAME3,ITEM_CODE4,ITEM_NAME4,UNIT_NAME,WGT,TIME,DATA_VALUE
0,817Y002,1.3.2.1. 시장금리(일별),010200000,국고채(3년),None,None,None,None,None,None,연%,None,20180205,2.287
1,817Y002,1.3.2.1. 시장금리(일별),010200000,국고채(3년),None,None,None,None,None,None,연%,None,20180206,2.253
2,817Y002,1.3.2.1. 시장금리(일별),010200000,국고채(3년),None,None,None,None,None,None,연%,None,20180207,2.248
3,817Y002,1.3.2.1. 시장금리(일별),010200000,국고채(3년),None,None,None,None,None,None,연%,None,20180208,2.272
4,817Y002,1.3.2.1. 시장금리(일별),010200000,국고채(3년),None,None,None,None,None,None,연%,None,20180209,2.278


In [16]:
#05b-16.usd/krw 및 국고채 3년 raw data 저장

usdkrw_raw.to_csv(
    ECOS_RAW_DIR / "usdkrw_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

bond3y_raw.to_csv(
    ECOS_RAW_DIR / "bond3y_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    "USD/KRW saved:",
    ECOS_RAW_DIR / "usdkrw_raw.csv"
)

print(
    "Bond 3Y saved:",
    ECOS_RAW_DIR / "bond3y_raw.csv"
)

USD/KRW saved: C:\code\portfolio_optimization\data\raw\ecos\usdkrw_raw.csv
Bond 3Y saved: C:\code\portfolio_optimization\data\raw\ecos\bond3y_raw.csv


In [17]:
#05b-17. usd/krw 및 국고채 3년 데이터 정제
#time->data, data_value->실제변수명

usdkrw_clean = (
    usdkrw_raw[
        [
            "TIME",
            "DATA_VALUE"
        ]
    ]
    .copy()
    .rename(
        columns={
            "TIME": "date",
            "DATA_VALUE": "usdkrw"
        }
    )
)

usdkrw_clean["date"] = pd.to_datetime(
    usdkrw_clean["date"],
    format="%Y%m%d"
)

usdkrw_clean["usdkrw"] = pd.to_numeric(
    usdkrw_clean["usdkrw"],
    errors="coerce"
)


bond3y_clean = (
    bond3y_raw[
        [
            "TIME",
            "DATA_VALUE"
        ]
    ]
    .copy()
    .rename(
        columns={
            "TIME": "date",
            "DATA_VALUE": "bond3y"
        }
    )
)

bond3y_clean["date"] = pd.to_datetime(
    bond3y_clean["date"],
    format="%Y%m%d"
)

bond3y_clean["bond3y"] = pd.to_numeric(
    bond3y_clean["bond3y"],
    errors="coerce"
)


usdkrw_clean = (
    usdkrw_clean
    .sort_values("date")
    .reset_index(drop=True)
)

bond3y_clean = (
    bond3y_clean
    .sort_values("date")
    .reset_index(drop=True)
)

In [18]:
#05b-18. 데이터 검증 / 환율, 국고채 정제 결과 확인
#목적: 기간, 행 수 , 결측, 중복 확인

print(
    "USD/KRW"
)

print(
    "Start:",
    usdkrw_clean["date"].min()
)

print(
    "End:",
    usdkrw_clean["date"].max()
)

print(
    "Rows:",
    len(usdkrw_clean)
)

print(
    "Missing:",
    usdkrw_clean["usdkrw"].isna().sum()
)

print(
    "Duplicates:",
    usdkrw_clean["date"].duplicated().sum()
)


print(
    "\nBond 3Y"
)

print(
    "Start:",
    bond3y_clean["date"].min()
)

print(
    "End:",
    bond3y_clean["date"].max()
)

print(
    "Rows:",
    len(bond3y_clean)
)

print(
    "Missing:",
    bond3y_clean["bond3y"].isna().sum()
)

print(
    "Duplicates:",
    bond3y_clean["date"].duplicated().sum()
)

USD/KRW
Start: 2018-02-05 00:00:00
End: 2026-09-15 00:00:00
Rows: 2114
Missing: 0
Duplicates: 0

Bond 3Y
Start: 2018-02-05 00:00:00
End: 2026-09-15 00:00:00
Rows: 2120
Missing: 0
Duplicates: 0


In [19]:
#05b-19. usd/krw 및 국고채 3년 clean data 저장
#목적: raw data와 정제 데이터 분리, 이후 05d common dataset에서 재사용

USDKRW_CLEAN_PATH = (
    ECOS_CLEAN_DIR
    / "usdkrw_clean.csv"
)

BOND3Y_CLEAN_PATH = (
    ECOS_CLEAN_DIR
    / "bond3y_clean.csv"
)

usdkrw_clean.to_csv(
    USDKRW_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

bond3y_clean.to_csv(
    BOND3Y_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "USD/KRW saved:",
    USDKRW_CLEAN_PATH
)

print(
    "Bond 3Y saved:",
    BOND3Y_CLEAN_PATH
)

USD/KRW saved: C:\code\portfolio_optimization\data\clean\ecos\usdkrw_clean.csv
Bond 3Y saved: C:\code\portfolio_optimization\data\clean\ecos\bond3y_clean.csv


In [20]:
#05b-20. 데이터 결합
#기준금리 + usd/krw + 국고채 3년 결합
#목적: macro data를 하나의 표로 관
#주의: 아직 krx trading calendar에 맞추지 않음, 아직 forward fill 하지 않음
#outer merge: 세 데이터 중 어느 하나에라도 존재하는 날짜는 전부 남김

macro_clean = (
    base_rate_clean
    .merge(
        usdkrw_clean,
        on="date",
        how="outer"
    )
    .merge(
        bond3y_clean,
        on="date",
        how="outer"
    )
    .sort_values("date")
    .reset_index(drop=True)
)

print(
    "Shape:",
    macro_clean.shape
)

macro_clean.head(10)

Shape: (3145, 4)


,date,base_rate,usdkrw,bond3y
0,2018-02-05,1.5,1088.5,2.287
1,2018-02-06,1.5,1091.5,2.253
2,2018-02-07,1.5,1086.6,2.248
3,2018-02-08,1.5,1087.9,2.272
4,2018-02-09,1.5,1092.1,2.278
5,2018-02-10,1.5,NaN,NaN
6,2018-02-11,1.5,NaN,NaN
7,2018-02-12,1.5,1084.6,2.302
8,2018-02-13,1.5,1084.5,2.278
9,2018-02-14,1.5,1077.2,2.265


In [21]:
#05b-21. macro 데이터 결합 후 결측 패턴 확인
#목적: 데이터 오류인지 시장별 관측일 차이 때문인지 확인

print(
    "Rows:",
    len(macro_clean)
)

print(
    "\nMissing values:"
)

print(
    macro_clean[
        [
            "base_rate",
            "usdkrw",
            "bond3y"
        ]
    ]
    .isna()
    .sum()
)

macro_clean.head(20)

Rows: 3145

Missing values:
base_rate       0
usdkrw       1031
bond3y       1025
dtype: int64


,date,base_rate,usdkrw,bond3y
0,2018-02-05,1.5,1088.5,2.287
1,2018-02-06,1.5,1091.5,2.253
2,2018-02-07,1.5,1086.6,2.248
3,2018-02-08,1.5,1087.9,2.272
4,2018-02-09,1.5,1092.1,2.278
5,2018-02-10,1.5,NaN,NaN
6,2018-02-11,1.5,NaN,NaN
7,2018-02-12,1.5,1084.6,2.302
8,2018-02-13,1.5,1084.5,2.278
9,2018-02-14,1.5,1077.2,2.265


In [22]:
#05b-22. 데이터 검증
#macro 변수별 실제 관측일 비교
#목적: 환율/채권/기준금리 calendar 차이 확인

macro_availability = pd.DataFrame({
    "variable": [
        "base_rate",
        "usdkrw",
        "bond3y"
    ],
    "observations": [
        macro_clean["base_rate"].notna().sum(),
        macro_clean["usdkrw"].notna().sum(),
        macro_clean["bond3y"].notna().sum()
    ],
    "first_date": [
        macro_clean.loc[
            macro_clean["base_rate"].notna(),
            "date"
        ].min(),

        macro_clean.loc[
            macro_clean["usdkrw"].notna(),
            "date"
        ].min(),

        macro_clean.loc[
            macro_clean["bond3y"].notna(),
            "date"
        ].min()
    ],
    "last_date": [
        macro_clean.loc[
            macro_clean["base_rate"].notna(),
            "date"
        ].max(),

        macro_clean.loc[
            macro_clean["usdkrw"].notna(),
            "date"
        ].max(),

        macro_clean.loc[
            macro_clean["bond3y"].notna(),
            "date"
        ].max()
    ]
})

macro_availability

,variable,observations,first_date,last_date
0,base_rate,3145,2018-02-05,2026-09-15
1,usdkrw,2114,2018-02-05,2026-09-15
2,bond3y,2120,2018-02-05,2026-09-15


In [23]:
#05b-23. ecos macro 통합 clean dataset 저장
#포함: base_rate: 한국은행 기준금리, usdkrw: 원/달러 환율 종가 15:30, body3y: 국고채 3년 금리
#주의: 아직 krx trading calendar 정렬 전임, 아직 결측치 forward fill 전

MACRO_CLEAN_PATH = (
    ECOS_CLEAN_DIR
    / "macro_daily_clean.csv"
)

macro_clean.to_csv(
    MACRO_CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Saved:",
    MACRO_CLEAN_PATH
)

Saved: C:\code\portfolio_optimization\data\clean\ecos\macro_daily_clean.csv


In [24]:
#05b-24. 05b 최종 검증
#ecos macro pipeline 결과 요약

print(
    "=== ECOS MACRO PIPELINE SUMMARY ==="
)

print(
    "\nBase Rate"
)

print(
    "Rows:",
    len(base_rate_clean)
)

print(
    "Range:",
    base_rate_clean["date"].min(),
    "~",
    base_rate_clean["date"].max()
)


print(
    "\nUSD/KRW"
)

print(
    "Rows:",
    len(usdkrw_clean)
)

print(
    "Range:",
    usdkrw_clean["date"].min(),
    "~",
    usdkrw_clean["date"].max()
)


print(
    "\nBond 3Y"
)

print(
    "Rows:",
    len(bond3y_clean)
)

print(
    "Range:",
    bond3y_clean["date"].min(),
    "~",
    bond3y_clean["date"].max()
)


print(
    "\nCombined Macro"
)

print(
    "Rows:",
    len(macro_clean)
)

print(
    "Duplicate dates:",
    macro_clean["date"].duplicated().sum()
)

print(
    "\nMissing values:"
)

print(
    macro_clean.isna().sum()
)

=== ECOS MACRO PIPELINE SUMMARY ===

Base Rate
Rows: 3145
Range: 2018-02-05 00:00:00 ~ 2026-09-15 00:00:00

USD/KRW
Rows: 2114
Range: 2018-02-05 00:00:00 ~ 2026-09-15 00:00:00

Bond 3Y
Rows: 2120
Range: 2018-02-05 00:00:00 ~ 2026-09-15 00:00:00

Combined Macro
Rows: 3145
Duplicate dates: 0

Missing values:
date            0
base_rate       0
usdkrw       1031
bond3y       1025
dtype: int64
